In [9]:
import pandas as pd
from datetime import timedelta
from pathlib import Path
import joblib



TEST_SIZE_DAYS = 30  # Last 30 days for testing
OUT_DIR = Path("../../data/FinalDataset")

In [10]:
df_features = pd.read_csv("../../data/FinalDataset/df_features.csv", parse_dates=['Date'])
df_processed = pd.read_csv("../../data/FinalDataset/df_processed.csv", parse_dates=['Date'])

In [11]:
# Keep ALL data - no dropping
# df_processed.drop(columns=["Station"], inplace=True)
df_clean = df_processed.copy()

# Ensure Date is datetime (CSV round-trip converts it to string)
df_clean['Date'] = pd.to_datetime(df_clean['Date'])

# Split based on date only
min_date = df_clean['Date'].min()  # 2023-01-01
max_date = df_clean['Date'].max()  # 2023-08-09
split_date = max_date - timedelta(days=TEST_SIZE_DAYS - 1)

# Split ALL data
train_df = df_clean[df_clean['Date'] < split_date].copy()
test_df = df_clean[df_clean['Date'] >= split_date].copy()

# For target columns: fill NaN with a special value or use masking
target_cols = [col for col in df_clean.columns if col.startswith('target_')]

# # Option 3a: Fill NaN targets with -1 (will be ignored during training)
# for col in target_cols:
#     train_df[col] = train_df[col].fillna(-1)
#     test_df[col] = test_df[col].fillna(-1)

# Option 3b: Create a mask to identify valid targets
# train_df['has_valid_target'] = ~train_df[f'target_{max(FORECAST_HORIZONS)}d_sum'].isna()

print(f"{'='*80}")
print(f"TRAIN-TEST SPLIT (ZERO Data Loss)")
print(f"{'='*80}")
print(f"\nData Range: {min_date.strftime('%Y-%m-%d')} to {max_date.strftime('%Y-%m-%d')}")
print(f"Split Date: {split_date.strftime('%Y-%m-%d')}")

print(f"\nTrain Set:")
print(f"  - Date range: {train_df['Date'].min().strftime('%Y-%m-%d')} to {train_df['Date'].max().strftime('%Y-%m-%d')}")
print(f"  - Shape: {train_df.shape}")
print(f"  - ALL {len(train_df):,} rows kept")

print(f"\nTest Set:")
print(f"  - Date range: {test_df['Date'].min().strftime('%Y-%m-%d')} to {test_df['Date'].max().strftime('%Y-%m-%d')}")
print(f"  - Shape: {test_df.shape}")
print(f"  - ALL {len(test_df):,} rows kept")

print(f"\nTotal: {len(df_clean):,} samples (100% data usage)")
print(f"Zero rows dropped!")
print(f"\n{'='*80}")

TRAIN-TEST SPLIT (ZERO Data Loss)

Data Range: 2023-01-01 to 2023-08-09
Split Date: 2023-07-11

Train Set:
  - Date range: 2023-01-01 to 2023-07-10
  - Shape: (7831, 130)
  - ALL 7,831 rows kept

Test Set:
  - Date range: 2023-07-11 to 2023-08-09
  - Shape: (1230, 130)
  - ALL 1,230 rows kept

Total: 9,061 samples (100% data usage)
Zero rows dropped!



In [12]:
train_df_with_date = train_df.copy()
test_df_with_date = test_df.copy()

In [13]:


OUT_DIR.mkdir(parents=True, exist_ok=True)

# CSV
train_df_with_date.to_csv(OUT_DIR / "train_df_with_date.csv", index=False)
test_df_with_date.to_csv(OUT_DIR / "test_df_with_date.csv", index=False)

# Parquet (best-effort)
try:
    train_df_with_date.to_parquet(OUT_DIR / "train_df_with_date.parquet", index=False)
    test_df_with_date.to_parquet(OUT_DIR / "test_df_with_date.parquet", index=False)
except Exception as e:
    print("Parquet save skipped or failed:", e)

# Pickle / joblib for quick reload
joblib.dump(
    {"train_df_with_date": train_df_with_date, "test_df_with_date": test_df_with_date},
    OUT_DIR / "train_test_with_date.joblib",
    compress=3
)

# Individual pickles as fallback
train_df_with_date.to_pickle(OUT_DIR / "train_df_with_date.pkl")
test_df_with_date.to_pickle(OUT_DIR / "test_df_with_date.pkl")

# Confirm saved files
for p in sorted(OUT_DIR.iterdir()):
    print(f"{p.name} - {p.stat().st_size/1024:.1f} KB")

Parquet save skipped or failed: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.
archive - 4.0 KB
df_features.csv - 2234.2 KB
df_processed.csv - 6995.1 KB
test_df_with_date.csv - 937.8 KB
test_df_with_date.pkl - 1256.4 KB
train_df_with_date.csv - 5941.8 KB
train_df_with_date.pkl - 7973.5 KB
train_test_with_date.joblib - 481.7 KB


<br> <br> <br>

### Prepare ML-Ready Datasets

In [14]:
# Define feature columns (exclude metadata and target columns)
exclude_cols = ["Sales"]  + [col for col in df_clean.columns if col.startswith('target_')] 
feature_cols = [col for col in df_clean.columns if col not in exclude_cols]

# Fill remaining NaN with 0 (from lag features at the beginning)
train_df[feature_cols] = train_df[feature_cols].fillna(0)
test_df[feature_cols] = test_df[feature_cols].fillna(0)

print(f"Feature columns ({len(feature_cols)}):")
print(f"  First 10: {feature_cols[:10]}")
print(f"  Last 10: {feature_cols[-10:]}")

print(f"\nTarget columns ({len([col for col in df_clean.columns if col.startswith('target_')])}):")
print(f"  {[col for col in df_clean.columns if col.startswith('target_')]}")

# Prepare X and y using target_1d_sum
print(f"\n{'='*80}")
print(f"ML-READY DATASETS PREPARED (Using target_1d_sum)")
print(f"{'='*80}")

# Use target_1d_sum for training/testing
# target_col = 'target_1d_sum'
target_col = 'Sales'

X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_test = test_df[feature_cols]
y_test = test_df[target_col]

print(f"\nTarget: {target_col}")
print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"  X_test: {X_test.shape}, y_test: {y_test.shape}")


# Check for -1 values (NaN replacements)
nan_count_train = (y_train == -1).sum()
nan_count_test = (y_test == -1).sum()
print(f"\n  NaN values (marked as -1):")
print(f"    Train: {nan_count_train} ({nan_count_train/len(y_train)*100:.1f}%)")
print(f"    Test: {nan_count_test} ({nan_count_test/len(y_test)*100:.1f}%)")

print(f"\n{'='*80}")
print(f"✓ Feature engineering pipeline complete!")
print(f"  Ready to train with: X_train, y_train, X_test, y_test")
print(f"{'='*80}")

Feature columns (129):
  First 10: ['Date', 'Production', 'Factory_Issue', 'Delivery', 'Station', 'Rainfall', 'Sunshine', 'Humidity', 'Temperature', 'typical_rain_mm']
  Last 10: ['category_religious_period', 'category_school_university', 'category_weather_event', 'scope_or_location_Chattogram division & south-east', 'scope_or_location_Nationwide', 'scope_or_location_Nationwide (strongest in Buddhist communities)', 'scope_or_location_Nationwide (strongest in cities)', 'scope_or_location_Tongi near Dhaka', 'scope_or_location_Urban centres (esp. Dhaka)', 'scope_or_location_nan']

Target columns (0):
  []

ML-READY DATASETS PREPARED (Using target_1d_sum)

Target: Sales
  X_train: (7831, 129), y_train: (7831,)
  X_test: (1230, 129), y_test: (1230,)

  NaN values (marked as -1):
    Train: 0 (0.0%)
    Test: 0 (0.0%)

✓ Feature engineering pipeline complete!
  Ready to train with: X_train, y_train, X_test, y_test


In [15]:
# Define feature columns (exclude metadata and target columns)
exclude_cols = ["Sales"]  + [col for col in df_clean.columns if col.startswith('target_')] 
feature_cols = [col for col in df_clean.columns if col not in exclude_cols]

# Fill remaining NaN with 0 (from lag features at the beginning)
train_df[feature_cols] = train_df[feature_cols].fillna(0)
test_df[feature_cols] = test_df[feature_cols].fillna(0)

print(f"Feature columns ({len(feature_cols)}):")
print(f"  First 10: {feature_cols[:10]}")
print(f"  Last 10: {feature_cols[-10:]}")

print(f"\nTarget columns ({len([col for col in df_clean.columns if col.startswith('target_')])}):")
print(f"  {[col for col in df_clean.columns if col.startswith('target_')]}")

# Prepare X and y using target_1d_sum
print(f"\n{'='*80}")
print(f"ML-READY DATASETS PREPARED (Using target_1d_sum)")
print(f"{'='*80}")

# Use target_1d_sum for training/testing
# target_col = 'target_1d_sum'
target_col = 'Sales'

X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_test = test_df[feature_cols]
y_test = test_df[target_col]

print(f"\nTarget: {target_col}")
print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"  X_test: {X_test.shape}, y_test: {y_test.shape}")


# Check for -1 values (NaN replacements)
nan_count_train = (y_train == -1).sum()
nan_count_test = (y_test == -1).sum()
print(f"\n  NaN values (marked as -1):")
print(f"    Train: {nan_count_train} ({nan_count_train/len(y_train)*100:.1f}%)")
print(f"    Test: {nan_count_test} ({nan_count_test/len(y_test)*100:.1f}%)")

print(f"\n{'='*80}")
print(f"✓ Feature engineering pipeline complete!")
print(f"  Ready to train with: X_train, y_train, X_test, y_test")
print(f"{'='*80}")

Feature columns (129):
  First 10: ['Date', 'Production', 'Factory_Issue', 'Delivery', 'Station', 'Rainfall', 'Sunshine', 'Humidity', 'Temperature', 'typical_rain_mm']
  Last 10: ['category_religious_period', 'category_school_university', 'category_weather_event', 'scope_or_location_Chattogram division & south-east', 'scope_or_location_Nationwide', 'scope_or_location_Nationwide (strongest in Buddhist communities)', 'scope_or_location_Nationwide (strongest in cities)', 'scope_or_location_Tongi near Dhaka', 'scope_or_location_Urban centres (esp. Dhaka)', 'scope_or_location_nan']

Target columns (0):
  []

ML-READY DATASETS PREPARED (Using target_1d_sum)

Target: Sales
  X_train: (7831, 129), y_train: (7831,)
  X_test: (1230, 129), y_test: (1230,)

  NaN values (marked as -1):
    Train: 0 (0.0%)
    Test: 0 (0.0%)

✓ Feature engineering pipeline complete!
  Ready to train with: X_train, y_train, X_test, y_test


In [16]:
X_train

,Date,Production,Factory_Issue,Delivery,Station,Rainfall,Sunshine,Humidity,Temperature,typical_rain_mm,...,category_religious_period,category_school_university,category_weather_event,scope_or_location_Chattogram division & south-east,scope_or_location_Nationwide,scope_or_location_Nationwide (strongest in Buddhist communities),scope_or_location_Nationwide (strongest in cities),scope_or_location_Tongi near Dhaka,scope_or_location_Urban centres (esp. Dhaka),scope_or_location_nan
0,2023-01-01,0.079538,0.049447,0.038181,Dhaka,0.000000,0.380952,0.777778,0.222222,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,2023-01-02,0.079538,0.087679,0.125128,Dhaka,0.000000,0.523810,0.619048,0.280193,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,2023-01-03,0.106050,0.151093,0.119652,Dhaka,0.000000,0.000000,0.873016,0.140097,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,2023-01-04,0.106050,0.115461,0.131717,Dhaka,0.000000,0.000000,0.936508,0.072464,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,2023-01-05,0.106050,0.056074,0.039736,Dhaka,0.000000,0.123810,0.857143,0.091787,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9026,2023-07-06,0.063630,0.059999,0.123994,Dhaka,0.026087,0.476190,0.777778,0.768116,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
9027,2023-07-07,0.000000,0.049192,0.062861,Dhaka,0.095652,0.495238,0.777778,0.782609,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
9028,2023-07-08,0.000000,0.041902,0.101528,Dhaka,0.095652,0.857143,0.666667,0.835749,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
9029,2023-07-09,0.079856,0.081817,0.109309,Dhaka,0.000000,0.161905,0.761905,0.816425,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


<br> <br> <br>

## Save X_train, X_test, y_train and y_test



In [17]:
from pathlib import Path
import os
import joblib


OUT_DIR.mkdir(parents=True, exist_ok=True)

# CSV exports
X_train.to_csv(OUT_DIR / "X_train.csv", index=True)
X_test.to_csv(OUT_DIR / "X_test.csv", index=True)
y_train.to_csv(OUT_DIR / "y_train.csv", index=True)
y_test.to_csv(OUT_DIR / "y_test.csv", index=True)

# Parquet exports (best-effort)
try:
    X_train.to_parquet(OUT_DIR / "X_train.parquet", index=True)
    X_test.to_parquet(OUT_DIR / "X_test.parquet", index=True)
    y_train.to_frame().to_parquet(OUT_DIR / "y_train.parquet", index=True)
    y_test.to_frame().to_parquet(OUT_DIR / "y_test.parquet", index=True)
except Exception as e:
    print("Parquet save skipped or failed:", e)

# Single binary file for quick reload
joblib.dump(
    {"X_train": X_train, "y_train": y_train, "X_test": X_test, "y_test": y_test},
    OUT_DIR / "final_dataset.joblib",
    compress=3
)

# Confirm saved files
for p in sorted(OUT_DIR.iterdir()):
    print(f"{p.name} - {p.stat().st_size/1024:.1f} KB")

Parquet save skipped or failed: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.
archive - 4.0 KB
df_features.csv - 2234.2 KB
df_processed.csv - 6995.1 KB
final_dataset.joblib - 540.4 KB
test_df_with_date.csv - 937.8 KB
test_df_with_date.pkl - 1256.4 KB
train_df_with_date.csv - 5941.8 KB
train_df_with_date.pkl - 7973.5 KB
train_test_with_date.joblib - 481.7 KB
X_test.csv - 937.4 KB
X_train.csv - 5945.2 KB
y_test.csv - 21.2 KB
y_train.csv - 137.3 KB
